# Coleta de notícias: caso Jensen Huang × NVIDIA
Período: 18/10/2026 a 18/11/2026, datas convertidas para o fuso de Nova York. Notícias em inglês coletadas via Google News RSS em janelas diárias. Gera a base bruta e as tabelas agregadas por dia de calendário e por pregão, usando a mesma regra de corte das 16h dos notebooks do YouTube e das ações.

In [1]:
%pip install feedparser

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\eaime\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import os, re, json, time, hashlib, pathlib, unicodedata, urllib.parse
import feedparser
import numpy as np
import pandas as pd


def raiz_do_projeto():
    # Acha a raiz do repositório (a pasta com .git), de onde quer que o notebook rode, em qualquer computador.
    candidatos = []
    if os.environ.get("PROJETO_DIR"):                 # opcional: força o caminho pela variável de ambiente PROJETO_DIR
        candidatos.append(pathlib.Path(os.environ["PROJETO_DIR"]))
    arq = globals().get("__vsc_ipynb_file__")         # caminho do próprio notebook, quando aberto no VS Code
    if arq:
        candidatos.append(pathlib.Path(arq).parent)
    candidatos.append(pathlib.Path.cwd())             # pasta de execução do kernel
    for c in candidatos:
        try:
            c = c.resolve()
        except OSError:
            continue
        for d in [c, *c.parents]:
            if (d / ".git").exists():
                return d
    raise RuntimeError("Não achei a raiz do projeto (pasta com .git). Abra o notebook de dentro do repositório "
                       f"(pasta notebooks/) ou defina PROJETO_DIR. Tentei: {[str(c) for c in candidatos]}")


RAIZ = raiz_do_projeto()
RAW, PROC = RAIZ / "data" / "raw", RAIZ / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

INICIO, FIM = "2026-10-18", "2026-11-18"   # dias no fuso de Nova York, ambos inclusos (igual ao YouTube)
FUSO = "America/New_York"
FECHAMENTO_BOLSA = 16                      # hora de NY em que a NYSE fecha
FERIADOS_BOLSA = []                        # feriados da NYSE (AAAA-MM-DD); nenhum entre 18/10 e 19/11/2026
ARQUIVO_MANUAL = None                      # plano B: nome de CSV em data/raw (colunas: data_publicacao, manchete, fonte, link)

CONSULTAS = [
    '"Jensen Huang"',
    '"Jensen Huang" NVIDIA',
    '"Jensen Huang" (spy OR affair OR resign OR scandal OR probe OR megaspeed)',
    'bringjensenback',
    # 'NVIDIA earnings',   # opcional: série de controle (balanço de 17/11). Só entra no `sobre_o_caso` se a manchete falar da pessoa/CEO.
]
TETO_RSS = 100                             # limite aproximado de itens por consulta no feed RSS do Google News


def salvar_json(caminho, obj):
    """Escrita atômica: se der erro no meio, o arquivo anterior continua intacto."""
    tmp = caminho.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8")
    tmp.replace(caminho)


def carregar_json(caminho, padrao):
    if not caminho.exists():
        return padrao
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        caminho.replace(caminho.with_suffix(".corrompido"))
        print(f"{caminho.name} estava corrompido; guardado como .corrompido e recomeçando vazio")
        return padrao


hash_busca = hashlib.md5(json.dumps([CONSULTAS, INICIO, FIM], sort_keys=True).encode()).hexdigest()[:8]
cache_arq = RAW / f"cache_noticias_{hash_busca}.json"
cache = carregar_json(cache_arq, {})

print("Raiz do projeto:", RAIZ)
print("Cache:", cache_arq.name)

Raiz do projeto: C:\Users\eaime\Downloads\jensen-huang-nvidia-analysis
Cache: cache_noticias_0b094fac.json


## Coleta via Google News RSS
Para evitar o teto de ~100 resultados por requisição do Google News RSS, o período é fatiado em janelas diárias com 1 dia de margem nas pontas (para não perder notícias de madrugada/noite após converter UTC para Nova York). Cada par `(consulta, dia)` fica salvo no cache em `data/raw/cache_noticias_*.json`.

In [3]:
def buscar_rss(query_completa, tentativas=4):
    q_enc = urllib.parse.quote(query_completa)
    url = f"https://news.google.com/rss/search?q={q_enc}&hl=en-US&gl=US&ceid=US:en"
    espera = 2
    for tentativa in range(tentativas):
        feed = feedparser.parse(url, agent="Mozilla/5.0 (UFPB-CienciaDados/1.0)")
        status = feed.get("status", 200)
        if status < 400 and not feed.get("bozo", False) or feed.entries:
            itens = []
            for e in feed.entries:
                fonte = e.source.title.strip() if hasattr(e, "source") and hasattr(e.source, "title") else "Desconhecida"
                itens.append({
                    "data_publicacao": e.get("published", ""),
                    "manchete_bruta": e.get("title", "").strip(),
                    "fonte": fonte,
                    "link": e.get("link", "").strip(),
                })
            return itens
        time.sleep(espera)
        espera *= 2
    raise RuntimeError(f"Falha ao consultar Google News RSS para: {query_completa}")


if ARQUIVO_MANUAL:
    bruto = pd.read_csv(RAW / ARQUIVO_MANUAL)
    if "manchete_bruta" not in bruto.columns and "manchete" in bruto.columns:
        bruto["manchete_bruta"] = bruto["manchete"]
    bruto["consulta_origem"] = "arquivo_manual"
    print(f"Carregado arquivo manual {ARQUIVO_MANUAL}: {len(bruto)} registros")
else:
    # Margem de 1 dia antes e depois para garantir cobertura completa após conversão UTC -> America/New_York
    dias_busca = pd.date_range(pd.Timestamp(INICIO) - pd.Timedelta(days=1),
                               pd.Timestamp(FIM) + pd.Timedelta(days=1))
    tarefas = [(c, d.strftime("%Y-%m-%d"), (d + pd.Timedelta(days=1)).strftime("%Y-%m-%d"))
               for c in CONSULTAS for d in dias_busca]
    pendentes = [t for t in tarefas if f"{t[0]}|{t[1]}" not in cache]
    print(f"{len(tarefas)} requisições totais ({len(CONSULTAS)} consultas × {len(dias_busca)} dias), {len(pendentes)} fora do cache")

    janelas_no_teto, janelas_vazias = [], []
    registros = []
    for consulta, d_ini, d_fim in tarefas:
        chave = f"{consulta}|{d_ini}"
        if chave not in cache:
            q_dia = f"{consulta} after:{d_ini} before:{d_fim}"
            cache[chave] = buscar_rss(q_dia)
            salvar_json(cache_arq, cache)
            time.sleep(0.4)
        itens = cache[chave]
        if len(itens) >= TETO_RSS:
            janelas_no_teto.append(chave)
        elif not itens:
            janelas_vazias.append(chave)
        for it in itens:
            registros.append({**it, "consulta_origem": consulta})

    if not registros:
        raise RuntimeError("Nenhuma notícia retornada: verifique a conexão ou use ARQUIVO_MANUAL.")
    bruto = pd.DataFrame(registros)
    bruto["data_coleta"] = pd.Timestamp.now(tz=FUSO).date()
    bruto.to_csv(RAW / "noticias_rss_bruto.csv", index=False)   # resposta do RSS antes de qualquer filtro (versione este arquivo: o Google News muda com o tempo)
    print(f"{len(bruto)} registros brutos coletados (salvos em data/raw/noticias_rss_bruto.csv)")
    print(f"janelas (consulta × dia) no teto de {TETO_RSS}: {len(janelas_no_teto)} | vazias: {len(janelas_vazias)} de {len(tarefas)}")
    if janelas_no_teto:
        print("  no teto (consulta|dia):", janelas_no_teto[:10], "..." if len(janelas_no_teto) > 10 else "")
    if janelas_vazias:
        print("  vazias podem ser dia sem notícia OU limite de requisições do Google; olhe as de dias úteis:", janelas_vazias[:10], "..." if len(janelas_vazias) > 10 else "")

136 requisições totais (4 consultas × 34 dias), 0 fora do cache
5625 registros brutos coletados (salvos em data/raw/noticias_rss_bruto.csv)
janelas (consulta × dia) no teto de 100: 37 | vazias: 27 de 136
  no teto (consulta|dia): ['"Jensen Huang"|2026-11-08', '"Jensen Huang"|2026-11-09', '"Jensen Huang"|2026-11-10', '"Jensen Huang"|2026-11-11', '"Jensen Huang"|2026-11-12', '"Jensen Huang"|2026-11-13', '"Jensen Huang"|2026-11-14', '"Jensen Huang"|2026-11-15', '"Jensen Huang"|2026-11-16', '"Jensen Huang"|2026-11-17'] ...
  vazias podem ser dia sem notícia OU limite de requisições do Google; olhe as de dias úteis: ['bringjensenback|2026-10-17', 'bringjensenback|2026-10-18', 'bringjensenback|2026-10-19', 'bringjensenback|2026-10-20', 'bringjensenback|2026-10-21', 'bringjensenback|2026-10-22', 'bringjensenback|2026-10-23', 'bringjensenback|2026-10-24', 'bringjensenback|2026-10-25', 'bringjensenback|2026-10-26'] ...


## Limpeza, fuso horário e classificação temática
1. **Limpeza do título:** o Google News RSS anexa `" - Nome do Veículo"` ao final da manchete; esse sufixo é removido para evitar falsos positivos na classificação por palavras-chave.
2. **Remoção de duplicados:** remove links idênticos e também notícias repetidas com mesma `(fonte, manchete_limpa, data)` vindas de consultas sobrepostas.
3. **Conversão de fuso e recorte:** converte `data_publicacao` para UTC (`hora_utc`) e para Nova York (`hora_nova_york`), extraindo `data` e filtrando estritamente `18/10/2026` a `18/11/2026`.
4. **Tema e caso (`sobre_o_caso`):** o filtro de tema usa `PESSOA` e `EMPRESA`, como no YouTube. `sobre_o_caso` usa a mesma regra do `coleta_youtube.ipynb`: termo exclusivo do caso (`CASO_FORTE`) ou pessoa/CEO no título junto com palavra de escândalo (`CASO_GERAL`). Isso evita marcar o ticker SPY e comunicados de escritórios de advocacia. `categoria_tema` é criada com palavras inteiras (antes, `"ai"` casava com `said`, `again` etc.).
5. **Calendário de pregão (`data_pregao`):** notícias publicadas em fim de semana ou a partir das 16h (NY) são atribuídas ao pregão seguinte. Salva `data/processed/noticias_limpas.csv` (o dado bruto do RSS fica em `data/raw/noticias_rss_bruto.csv`, salvo na etapa de coleta).

In [4]:
bdia = pd.offsets.CustomBusinessDay(holidays=FERIADOS_BOLSA)

PESSOA = re.compile(r"jens[eo]n\s+huang", re.I)
EMPRESA = re.compile(r"nvidia", re.I)

# Regra de "sobre_o_caso" (a MESMA nos notebooks do YouTube e das notícias).
# Um título é do caso se tem um termo exclusivo do caso (CASO_FORTE) OU se fala da pessoa/CEO (PESSOA_LARGA)
# junto com uma palavra de escândalo (CASO_GERAL). Exigir a pessoa evita falsos positivos como o ticker "SPY"
# ("NVDA, SPY e QQQ") e comunicados de escritórios de advocacia ("investigation on behalf of investors").
# "intelligence" sozinho NÃO entra: pegaria "artificial intelligence".
PESSOA_LARGA = re.compile(r"jens[eo]n\s+huang|\bhuang\b|\bceo\b|chief executive|nvidia(?:'s|’s)?\s+founder", re.I)
CASO_FORTE = re.compile(
    r"megaspeed|alice\s+huang|bring\s*jensen\s*back|chinese intelligence|"
    r"intelligence (?:asset|operative|agent|operation|effort)|espionage|subpoena", re.I)
CASO_GERAL = re.compile(
    r"\bspy\b|\bspies\b|\bspying\b|\baffair\b|resign|steps? down|stepping down|\bousted?\b|"
    r"scandal|investigat|interim ceo", re.I)


def eh_do_caso(titulos):
    t = titulos.astype(str)
    return (t.str.contains(CASO_FORTE) | (t.str.contains(PESSOA_LARGA) & t.str.contains(CASO_GERAL))).astype(bool)


def remover_sufixo_fonte(row):
    titulo = str(row["manchete_bruta"]).strip()
    fonte = str(row["fonte"]).strip()
    sufixo = f" - {fonte}"
    if fonte and titulo.endswith(sufixo):
        return titulo[:-len(sufixo)].strip()
    return titulo


def limpar_texto(t):
    t = unicodedata.normalize("NFKD", str(t))
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^\w\s]", " ", t.lower())
    return re.sub(r"\s+", " ", t).strip()


def origem_tema(titulo):
    if PESSOA.search(titulo):
        return "manchete_pessoa"
    if EMPRESA.search(titulo):
        return "manchete_empresa"
    return None


def tem_termo(t, termos):
    """Palavra inteira; termo terminado em * casa o início da palavra (ex.: 'export*' casa 'exports')."""
    for p in termos:
        padrao = rf"\b{re.escape(p[:-1])}" if p.endswith("*") else rf"\b{re.escape(p)}\b"
        if re.search(padrao, t):
            return True
    return False


def classificar_categoria(row):
    if row["sobre_o_caso"]:
        return "Caso e Governanca"
    t = row["manchete_limpa"]
    if tem_termo(t, ["china", "export*", "tariff*", "ban", "sanction*", "us government", "washington", "beijing"]):
        return "Geopolitica e Regulacao"
    if tem_termo(t, ["stock*", "share*", "earnings", "revenue*", "market*", "trillion*", "wall street", "nasdaq", "investor*"]):
        return "Mercado e Acoes"
    if tem_termo(t, ["ai", "blackwell", "rubin", "chip*", "gpu*", "cuda", "data center*", "robot*", "gtc"]):
        return "Tecnologia e Produtos"
    return "Outros"


def data_pregao(ts):
    """Pregão que recebe a notícia: o mesmo dia (útil, antes das 16h NY) ou o próximo dia útil."""
    d = pd.Timestamp(ts.date())
    if ts.hour >= FECHAMENTO_BOLSA or not bdia.is_on_offset(d):
        return (d + bdia).date()
    return d.date()


funil = {"registros_brutos": len(bruto)}

df = bruto.copy()
df["manchete"] = df.apply(remover_sufixo_fonte, axis=1)
df["manchete_limpa"] = df["manchete"].apply(limpar_texto)
df["fonte"] = df["fonte"].fillna("Desconhecida").str.strip()

# Remove valores ausentes essenciais e duplicados por link
df = df[df["manchete_limpa"].str.len() > 0].copy()
df = df.drop_duplicates("link").copy()
funil["duplicados_link"] = funil["registros_brutos"] - len(df)

# Conversão de datas para UTC e fuso da NYSE
df["hora_utc"] = pd.to_datetime(df["data_publicacao"], utc=True, errors="coerce")
sem_data = int(df["hora_utc"].isna().sum())
funil["sem_data_valida"] = sem_data
df = df[df["hora_utc"].notna()].copy()

df["hora_nova_york"] = df["hora_utc"].dt.tz_convert(FUSO)
df["data"] = df["hora_nova_york"].dt.date

# Corte estrito do período do estudo (18/10 a 18/11/2026)
no_periodo = (df["data"] >= pd.to_datetime(INICIO).date()) & (df["data"] <= pd.to_datetime(FIM).date())
funil["fora_do_periodo"] = int((~no_periodo).sum())
df = df[no_periodo].copy()

# Deduplicação por mesma notícia (mesma fonte, manchete normalizada e dia)
antes_dedup = len(df)
df = df.sort_values("hora_utc").drop_duplicates(subset=["fonte", "manchete_limpa", "data"], keep="first").copy()
funil["duplicados_manchete_fonte"] = antes_dedup - len(df)

# Filtro de aderência ao tema (cita Jensen Huang ou NVIDIA na manchete)
df["tema_origem"] = df["manchete"].apply(origem_tema)
funil["fora_do_tema"] = int(df["tema_origem"].isna().sum())
df = df[df["tema_origem"].notna()].copy()

if df.empty:
    raise RuntimeError("Nenhuma notícia sobrou após os filtros.")

# Feature Engineering
df["sobre_o_caso"] = eh_do_caso(df["manchete"])
df["categoria_tema"] = df.apply(classificar_categoria, axis=1)
df["dia_semana"] = df["hora_nova_york"].dt.day_name()
df["hora_publicacao"] = df["hora_nova_york"].dt.hour
df["data_pregao"] = df["hora_nova_york"].apply(data_pregao)
df["data_coleta"] = pd.Timestamp.now(tz=FUSO).date()
df["noticia_id"] = df.apply(
    lambda r: hashlib.md5(f"{r['fonte']}|{r['manchete_limpa']}|{r['data']}".encode()).hexdigest()[:12],
    axis=1,
)

df = df.sort_values("hora_utc").reset_index(drop=True)
funil["noticias_finais"] = len(df)

colunas_raw = [
    "noticia_id", "hora_utc", "hora_nova_york", "data", "data_pregao",
    "dia_semana", "hora_publicacao", "fonte", "manchete", "manchete_limpa",
    "tema_origem", "sobre_o_caso", "categoria_tema", "link", "data_coleta",
]
df = df[colunas_raw]
df.to_csv(PROC / "noticias_limpas.csv", index=False)

print(pd.Series(funil).to_string())
print(f"\nNotícias sobre_o_caso: {int(df['sobre_o_caso'].sum())} de {len(df)}")
print("Deslocadas para o pregão seguinte:", int((df["data_pregao"] != df["data"]).sum()))

registros_brutos             5625
duplicados_link              3381
sem_data_valida                 0
fora_do_periodo               180
duplicados_manchete_fonte       7
fora_do_tema                  212
noticias_finais              1845

Notícias sobre_o_caso: 711 de 1845
Deslocadas para o pregão seguinte: 640


## Agregação por dia e por pregão
Gera as quatro tabelas agregadas com exatamente o mesmo calendário do `coleta_youtube.ipynb`, preenchendo dias sem notícias com zero para que os cruzamentos (`merge`) não percam datas:
- `noticias_diario.csv` e `noticias_diario_caso.csv`: 32 dias de calendário (`18/10` a `18/11/2026`).
- `noticias_diario_pregao.csv` e `noticias_diario_pregao_caso.csv`: pregões da NYSE com a mesma flag `cobertura_completa` nas pontas.

In [5]:
def agregar_noticias(base, coluna, calendario):
    if base.empty:
        g = pd.DataFrame(index=calendario, columns=[
            "n_noticias", "n_fontes_unicas", "n_caso", "n_mercado", "n_tecnologia", "n_geopolitica"
        ]).fillna(0)
    else:
        tmp = base.copy()
        tmp["is_caso"] = tmp["sobre_o_caso"].astype(int)
        tmp["is_mercado"] = (tmp["categoria_tema"] == "Mercado e Acoes").astype(int)
        tmp["is_tecnologia"] = (tmp["categoria_tema"] == "Tecnologia e Produtos").astype(int)
        tmp["is_geopolitica"] = (tmp["categoria_tema"] == "Geopolitica e Regulacao").astype(int)

        g = tmp.groupby(coluna).agg(
            n_noticias=("noticia_id", "count"),
            n_fontes_unicas=("fonte", "nunique"),
            n_caso=("is_caso", "sum"),
            n_mercado=("is_mercado", "sum"),
            n_tecnologia=("is_tecnologia", "sum"),
            n_geopolitica=("is_geopolitica", "sum"),
        )
        g.index = pd.to_datetime(g.index)
        g = g.reindex(calendario, fill_value=0)

    g["prop_caso"] = (g["n_caso"] / g["n_noticias"].where(g["n_noticias"] > 0)).round(4)
    g.index.name = coluna
    return g.reset_index()


# 1. Por dia de calendário (NY)
cal_dias = pd.date_range(INICIO, FIM)
diario = agregar_noticias(df, "data", cal_dias)
diario["dia_util"] = diario["data"].dt.dayofweek < 5
diario["data"] = diario["data"].dt.date
diario.to_csv(PROC / "noticias_diario.csv", index=False)

caso = df[df["sobre_o_caso"]]
diario_caso = agregar_noticias(caso, "data", cal_dias)
diario_caso["dia_util"] = diario_caso["data"].dt.dayofweek < 5
diario_caso["data"] = diario_caso["data"].dt.date
diario_caso.to_csv(PROC / "noticias_diario_caso.csv", index=False)

# 2. Por pregão da NYSE (mesma regra do coleta_youtube.ipynb)
ultimo = max(pd.Timestamp(FIM), pd.Timestamp(df["data_pregao"].max()))
pregoes = pd.bdate_range(INICIO, ultimo, freq="C", holidays=FERIADOS_BOLSA)

diario_pregao = agregar_noticias(df, "data_pregao", pregoes)
anterior = pd.Series([d - bdia for d in diario_pregao["data_pregao"]])
diario_pregao["cobertura_completa"] = ((anterior >= pd.Timestamp(INICIO)).values
                                       & (diario_pregao["data_pregao"] <= pd.Timestamp(FIM)).values)
diario_pregao["data_pregao"] = diario_pregao["data_pregao"].dt.date
diario_pregao.to_csv(PROC / "noticias_diario_pregao.csv", index=False)

diario_pregao_caso = agregar_noticias(caso, "data_pregao", pregoes)
diario_pregao_caso["cobertura_completa"] = diario_pregao["cobertura_completa"].values
diario_pregao_caso["data_pregao"] = diario_pregao_caso["data_pregao"].dt.date
diario_pregao_caso.to_csv(PROC / "noticias_diario_pregao_caso.csv", index=False)

print(len(diario), "dias |", int(diario["dia_util"].sum()), "dias úteis |",
      len(diario_pregao), "pregões,", int(diario_pregao["cobertura_completa"].sum()), "com cobertura completa")
print(len(caso), "notícias sobre o caso em", int((diario_caso["n_noticias"] > 0).sum()), "dias")

32 dias | 23 dias úteis | 24 pregões, 22 com cobertura completa
711 notícias sobre o caso em 23 dias


### Conferência e amostras de manchetes

In [6]:
print(df["data"].min(), "->", df["data"].max())
print("Valores ausentes nas colunas principais:\n", df[["data", "data_pregao", "fonte", "manchete"]].isna().sum())
print("\nDistribuição por categoria_tema:")
print(df["categoria_tema"].value_counts().to_string())
print("\nTop 10 veículos (fontes):")
print(df["fonte"].value_counts().head(10).to_string())

for rotulo, mascara in [("MARCADAS (sobre_o_caso = True)", df["sobre_o_caso"]),
                        ("NÃO MARCADAS (amostra)", ~df["sobre_o_caso"])]:
    sub = df[mascara]
    print(f"\n{rotulo}:")
    print(sub.sample(min(10, len(sub)), random_state=0)[["data", "fonte", "manchete"]].to_string(index=False, max_colwidth=65))

diario_pregao

2026-10-18 -> 2026-11-18
Valores ausentes nas colunas principais:
 data           0
data_pregao    0
fonte          0
manchete       0
dtype: int64

Distribuição por categoria_tema:
categoria_tema
Outros                     711
Caso e Governanca          711
Tecnologia e Produtos      186
Mercado e Acoes            160
Geopolitica e Regulacao     77

Top 10 veículos (fontes):
fonte
Yahoo Finance         134
CNBC                   99
Business Insider       63
The New York Times     58
WSJ                    47
Forbes                 38
New York Post          38
Bloomberg.com          36
TechCrunch             36
CNN                    32

MARCADAS (sobre_o_caso = True):
      data               fonte                                                          manchete
2026-11-10    Business Insider Jensen Huang is sliding down the global rich list, and his aff...
2026-11-05             Reuters Key Republicans seek briefing from Nvidia CEO over Chinese spy...
2026-11-08        The Guardian 


NÃO MARCADAS (amostra):
      data                   fonte                                                          manchete
2026-11-11      The New York Times Tech Allies Rally Behind Jensen Huang, Calling Him Vital in A....
2026-11-18                PC Gamer Jeff Bezos thinks 'Nvidia is too complicated' for an outside C...
2026-11-11                CBS News Nvidia's stock dives as chipmaker's board weighs move to oust ...
2026-11-13           Yahoo Finance Before His Ouster, Jensen Huang Belittled the Executive Now Ru...
2026-11-12                     CNN The $1 trillion question facing Nvidia and Jensen Huang: Who l...
2026-11-08                     WSJ Jensen Huang Apologized to Nvidia for His Infidelity. His Kids...
2026-10-19             eu.36kr.com Jensen Huang: The rise of TSMC has never been achieved by work...
2026-11-15           Bloomberg.com    Jensen Huang Comeback Report Primes Nvidia Shares for a Bounce
2026-10-21 Business Insider Africa Nvidia marketing executive Laur

,data_pregao,n_noticias,n_fontes_unicas,n_caso,n_mercado,n_tecnologia,n_geopolitica,prop_caso,cobertura_completa
0,2026-10-19,36,32,1,3,17,1,0.0278,False
1,2026-10-20,7,7,0,0,4,0,0.0000,True
2,2026-10-21,35,22,1,5,23,0,0.0286,True
3,2026-10-22,21,15,0,4,7,1,0.0000,True
4,2026-10-23,48,41,0,4,24,2,0.0000,True
5,2026-10-26,63,46,0,13,38,3,0.0000,True
6,2026-10-27,11,8,0,2,2,2,0.0000,True
7,2026-10-28,1,1,0,1,0,0,0.0000,True
8,2026-10-29,36,28,26,1,1,3,0.7222,True
9,2026-10-30,76,65,50,2,4,6,0.6579,True


### Validação dos CSVs e teste de junção com YouTube e Ações

In [7]:
salvo = pd.read_csv(PROC / "noticias_limpas.csv")
d_salvo = pd.read_csv(PROC / "noticias_diario.csv")
p_salvo = pd.read_csv(PROC / "noticias_diario_pregao.csv")
dc_salvo = pd.read_csv(PROC / "noticias_diario_caso.csv")
pc_salvo = pd.read_csv(PROC / "noticias_diario_pregao_caso.csv")

print("notícias:", len(salvo), "| dias:", len(d_salvo), "| pregões:", len(p_salvo))
print("noticia_id duplicados:", int(salvo["noticia_id"].duplicated().sum()))
print("soma diário == total notícias:", len(salvo) == int(d_salvo["n_noticias"].sum()))
print("soma pregão == total notícias:", len(salvo) == int(p_salvo["n_noticias"].sum()))
print("soma diário (caso) == sobre_o_caso:", int(salvo["sobre_o_caso"].sum()) == int(dc_salvo["n_noticias"].sum()))
print("soma pregão (caso) == sobre_o_caso:", int(salvo["sobre_o_caso"].sum()) == int(pc_salvo["n_noticias"].sum()))

ny = pd.to_datetime(salvo["hora_utc"], utc=True).dt.tz_convert(FUSO)
print("data confere com hora_utc convertida:", bool((ny.dt.strftime("%Y-%m-%d") == salvo["data"]).all()))
print("data_pregao sempre em dia útil:", bool((pd.to_datetime(salvo["data_pregao"]).dt.dayofweek < 5).all()))
print("data_pregao nunca antes de data:", bool((salvo["data_pregao"] >= salvo["data"]).all()))

# Teste de junção integrada das 3 bases (Notícias + YouTube + Ações)
arq_acoes = PROC / "acoes_nvidia.csv"
arq_yt = PROC / "youtube_diario_pregao.csv"
if arq_acoes.exists() and arq_yt.exists():
    ac = pd.read_csv(arq_acoes)
    yt = pd.read_csv(arq_yt)
    # cobertura_completa existe nas duas tabelas: renomeia antes de juntar, para não virar _x/_y
    yt = yt.rename(columns={"cobertura_completa": "cobertura_completa_yt"})
    nt = p_salvo.rename(columns={"cobertura_completa": "cobertura_completa_noticias"})
    integrado = ac.merge(yt, on="data_pregao", how="inner").merge(nt, on="data_pregao", how="inner")
    print(f"\nJunção integrada (Ações + YouTube + Notícias): {len(integrado)} pregões em comum")
    print(integrado[["data_pregao", "nvda_variacao_pct", "retorno_anormal_pp", "n_videos", "views", "n_noticias", "n_caso"]].head(10).to_string(index=False))

notícias: 1845 | dias: 32 | pregões: 24
noticia_id duplicados: 0
soma diário == total notícias: True
soma pregão == total notícias: True
soma diário (caso) == sobre_o_caso: True
soma pregão (caso) == sobre_o_caso: True
data confere com hora_utc convertida: True
data_pregao sempre em dia útil: True
data_pregao nunca antes de data: True

Junção integrada (Ações + YouTube + Notícias): 23 pregões em comum
data_pregao  nvda_variacao_pct  retorno_anormal_pp  n_videos    views  n_noticias  n_caso
 2026-10-19             1.3823              0.1764       113  2992871          36       1
 2026-10-20            -0.2856             -0.1840        67  1195909           7       0
 2026-10-21             2.5408              1.4859        82  1065293          35       1
 2026-10-22            -4.3970             -1.5385       119 11970196          21       0
 2026-10-23            -0.5890             -0.5791       109 22154053          48       0
 2026-10-26             2.3891              1.2818     

## Documentação da fonte e limitações (para o relatório)
**Fonte:** Google News RSS (`https://news.google.com/rss/search`), acessado na data da coluna `data_coleta` em fuso `America/New_York`, cobrindo o período de `18/10/2026` a `18/11/2026`.

**Por que junto com as outras bases:** `data_pregao` usa exatamente a mesma regra de calendário e horário de fechamento (16h de Nova York) das tabelas `youtube_diario_pregao.csv` e `acoes_nvidia.csv`, permitindo cruzar na mesma linha a cobertura da imprensa, o engajamento em vídeo e o retorno da ação `NVDA`.

**Limitações:**
1. O Google News RSS retorna uma amostra ordenada por relevância limitada a cerca de 100 itens por consulta/dia; o fatiamento em janelas diárias com múltiplas consultas reduz esse teto, mas o total de manchetes representa a cobertura indexada pela plataforma, não a totalidade da imprensa mundial.
2. Algumas manchetes não trazem o horário exato do veículo original, apenas o instante de indexação no feed RSS, o que pode deslocar notícias publicadas perto das 16h (NY) para o pregão seguinte.
3. A classificação `sobre_o_caso` e `categoria_tema` baseia-se em regras determinísticas de palavras-chave sobre a manchete. `sobre_o_caso` exige a pessoa/CEO no título (ou um termo exclusivo do caso), então pode deixar de marcar manchetes do caso que falem só de "Nvidia" e de capturar títulos metafóricos ou neutros; em troca, evita falsos positivos. Confira as amostras da seção de conferência.
4. A base só contém manchetes encontradas pelas consultas sobre Jensen Huang. `n_mercado`, `n_tecnologia` e `n_geopolitica` não medem a cobertura geral da NVIDIA; para controlar eventos como o balanço de 17/11, ative a consulta opcional `NVIDIA earnings` em `CONSULTAS`.
5. O Google News muda com o tempo e não dá para refazer a coleta igual depois. Por isso o dado bruto (`data/raw/noticias_rss_bruto.csv`) deve ser versionado no repositório.